# Textbook task data audit
Grain: one independently reviewed textbook task per ID. This notebook checks the safe frozen manifest, not private question bodies. Source-level uniqueness, referential integrity and split checks are reproducible in `scripts/prepare_grounded_supply_training.py` and its tests. One ambiguous author-specific framework question was held out after an eight-task spot check. No full manual quality guarantee or training-success claim follows from these counts. No time-series comparisons apply to this single frozen batch.

In [ ]:
import json
from pathlib import Path
folder = Path.cwd() if Path('supply_chain_task_training_data_20260909.safe.json').exists() else Path('docs')
m = json.loads((folder / 'supply_chain_task_training_data_20260909.safe.json').read_text())
assert m['input_count'] == m['retained'] + len(m['rejected']) == 697
assert m['train']['count'] + m['dev']['count'] == m['retained'] == 696
for split in ('train', 'dev'):
    s = m[split]
    assert sum(s['kind_counts'].values()) == sum(s['domain_counts'].values()) == s['count']
    assert s['objective_cases'] == s['count'] - s['kind_counts']['short_answer']
    assert s['max_length'] <= 4096 and s['loss_tokens'] < s['sequence_tokens']
assert m['steps_one_epoch'] * m['batch_size'] + m['dropped_tail_one_epoch'] == m['train']['count']
print({s: {'tasks': m[s]['count'], 'concepts': m[s]['concepts'], 'objective_cases': m[s]['objective_cases']} for s in ('train','dev')})
print('Held fraction:', len(m['rejected']) / m['input_count'])

## Interpretation
591 training tasks and 105 development tasks cover 199 and 35 primary units. Only 74 development questions have objective set-valued scoring; 31 free answers must not be silently included in that accuracy denominator. Development chapters were excluded from new SFT but were seen in book CPT. Topic coverage remains uneven (only 27 material-handling training tasks), so source breadth is a limitation. Tests block cross-split source references and contradictory duplicate labels; exact full-input duplicates are removed with development priority. Safe hashes retain lineage; all private QA and benchmark content remain on the server.